# 03 — Feature Engineering

Builds the modelling-ready feature set from `merged_players.parquet`.

**Golden rule:** every feature for gameweek N uses only data from gameweeks 1..N-1.
All rolling calculations use `.shift(1)` before `.rolling()` to enforce this.

**Output:** `data/processed/features.parquet`

## 1. Imports & load data

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path

RAW       = Path('../data/raw')
PROCESSED = Path('../data/processed')

df = pd.read_parquet(PROCESSED / 'merged_players.parquet')
print('Shape:', df.shape)
print('Rounds:', sorted(df['round'].unique()))

## 2. Data quality fixes

ICT columns were stored as strings. `xG/xA` nulls mean no Understat match — fill with 0 (correct prior: no shot data = 0 contribution).

In [ ]:
# Cast ICT columns to float
for col in ['influence', 'creativity', 'threat', 'ict_index']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Parse kickoff_time
df['kickoff_time'] = pd.to_datetime(df['kickoff_time'], utc=True)

# Fill Understat nulls with 0
xg_cols = ['xG', 'xA', 'shots', 'key_passes', 'npxG', 'xGChain', 'xGBuildup', 'us_minutes']
df[xg_cols] = df[xg_cols].fillna(0)

# Sort — critical for correct rolling calculations
df = df.sort_values(['player_id', 'round']).reset_index(drop=True)

print('Dtypes fixed. Null check:')
print(df[['influence', 'xG', 'xA']].isna().sum())

## 3. Form features (rolling FPL points + minutes)

`.shift(1)` moves each row's value one position forward within the group,
so `rolling(3).mean()` at row N uses rows N-3..N-1, never row N itself.

In [ ]:
def rolling_mean(group, col, window):
    return group[col].shift(1).rolling(window, min_periods=1).mean()

def rolling_std(group, col, window):
    return group[col].shift(1).rolling(window, min_periods=2).std()

# Points rolling averages
df['rolling_pts_3gw'] = df.groupby('player_id', group_keys=False).apply(
    lambda g: rolling_mean(g, 'total_points', 3)
)
df['rolling_pts_5gw'] = df.groupby('player_id', group_keys=False).apply(
    lambda g: rolling_mean(g, 'total_points', 5)
)

# Minutes rolling averages
df['rolling_minutes_3gw'] = df.groupby('player_id', group_keys=False).apply(
    lambda g: rolling_mean(g, 'minutes', 3)
)
df['rolling_minutes_5gw'] = df.groupby('player_id', group_keys=False).apply(
    lambda g: rolling_mean(g, 'minutes', 5)
)

# Minutes consistency — std dev (high = rotation risk)
df['minutes_consistency'] = df.groupby('player_id', group_keys=False).apply(
    lambda g: rolling_std(g, 'minutes', 5)
).fillna(0)

# Blank GW flag — did the player not play last GW?
df['prev_minutes'] = df.groupby('player_id')['minutes'].shift(1)
df['blank_gw_flag'] = (df['prev_minutes'] == 0).astype(int)
df.drop(columns=['prev_minutes'], inplace=True)

# Form streak — consecutive GWs with pts > 4
def form_streak(group):
    pts = group['total_points'].shift(1)
    streak = []
    count = 0
    for p in pts:
        if pd.isna(p):
            streak.append(0)
        elif p > 4:
            count += 1
            streak.append(count)
        else:
            count = 0
            streak.append(0)
    return pd.Series(streak, index=group.index)

df['form_streak'] = df.groupby('player_id', group_keys=False).apply(form_streak)

print('Form features done.')
print(df[['player_id', 'round', 'total_points', 'rolling_pts_3gw', 'rolling_pts_5gw',
          'rolling_minutes_3gw', 'minutes_consistency', 'blank_gw_flag', 'form_streak']].head(10).to_string(index=False))

In [ ]:
# Sanity check: rolling_pts_3gw at round 4 should equal mean of rounds 1-3
pid = df['player_id'].iloc[0]
player = df[df['player_id'] == pid][['round', 'total_points', 'rolling_pts_3gw']].head(6)
print(f'Player {pid} — verify no leakage:')
print(player.to_string(index=False))
# rolling_pts_3gw at round 4 should equal mean of rows 1,2,3 total_points

## 4. xG/xA rolling features

In [ ]:
for stat, windows in [('xG', [3, 5]), ('xA', [3, 5])]:
    for w in windows:
        col_name = f'rolling_{stat.lower()}_{w}gw'
        df[col_name] = df.groupby('player_id', group_keys=False).apply(
            lambda g, s=stat, ww=w: rolling_mean(g, s, ww)
        )

# xG overperformance: goals scored minus xG (rolling 5 GWs)
# Positive = finishing above expectation (likely to regress)
# Negative = unlucky, due goals
df['goals_minus_xg'] = df['goals_scored'] - df['xG']
df['xg_overperformance'] = df.groupby('player_id', group_keys=False).apply(
    lambda g: rolling_mean(g, 'goals_minus_xg', 5)
).fillna(0)
df.drop(columns=['goals_minus_xg'], inplace=True)

# Per-90 metrics (use rolling minutes to normalise)
# Add small epsilon to avoid division by zero
eps = 1e-6
df['shots_per_90'] = (
    df.groupby('player_id', group_keys=False).apply(lambda g: rolling_mean(g, 'shots', 5))
    / (df.groupby('player_id', group_keys=False).apply(lambda g: rolling_mean(g, 'minutes', 5)) / 90 + eps)
).fillna(0)

df['key_passes_per_90'] = (
    df.groupby('player_id', group_keys=False).apply(lambda g: rolling_mean(g, 'key_passes', 5))
    / (df.groupby('player_id', group_keys=False).apply(lambda g: rolling_mean(g, 'minutes', 5)) / 90 + eps)
).fillna(0)

print('xG/xA features done.')
xg_feat_cols = ['player_id', 'round', 'xG', 'rolling_xg_3gw', 'rolling_xg_5gw',
                'xA', 'rolling_xa_3gw', 'rolling_xa_5gw', 'xg_overperformance',
                'shots_per_90', 'key_passes_per_90']
print(df[xg_feat_cols].head(10).to_string(index=False))

## 5. Fixture features

For each player in each gameweek, look up the FDR and home/away status
for the **next** gameweek's fixture.

In [ ]:
fixtures = pd.read_parquet(RAW / 'fpl_fixtures.parquet')
teams    = pd.read_parquet(RAW / 'fpl_teams.parquet')

# Build a lookup: (team_id, gameweek) -> (fdr, is_home, opponent_id)
# Home perspective
home = fixtures[['event', 'team_h', 'team_a', 'team_h_difficulty', 'team_a_difficulty']].copy()
home = home.rename(columns={
    'event': 'round',
    'team_h': 'team_id',
    'team_a': 'opponent_id',
    'team_h_difficulty': 'fdr',
})
home['is_home'] = 1
home = home[['round', 'team_id', 'opponent_id', 'fdr', 'is_home']]

# Away perspective
away = fixtures[['event', 'team_a', 'team_h', 'team_a_difficulty']].copy()
away = away.rename(columns={
    'event': 'round',
    'team_a': 'team_id',
    'team_h': 'opponent_id',
    'team_a_difficulty': 'fdr',
})
away['is_home'] = 0
away = away[['round', 'team_id', 'opponent_id', 'fdr', 'is_home']]

fixture_lookup = pd.concat([home, away], ignore_index=True)
print('Fixture lookup shape:', fixture_lookup.shape)
fixture_lookup.head()

In [ ]:
# For each row, we want the fixture in the NEXT gameweek
# So we shift the round number by +1 when merging
next_fixture = fixture_lookup.copy()
next_fixture = next_fixture.rename(columns={
    'round': 'next_round',
    'fdr': 'fdr_next',
    'is_home': 'is_home_next',
    'opponent_id': 'next_opponent_id',
})

# Create a 'next_round' key on the main df
df['next_round'] = df['round'] + 1

df = df.merge(
    next_fixture[['next_round', 'team_id', 'fdr_next', 'is_home_next', 'next_opponent_id']],
    left_on=['next_round', 'team'],
    right_on=['next_round', 'team_id'],
    how='left',
).drop(columns=['team_id'])

# has_fixture flag: 0 for blank gameweeks
df['has_fixture'] = df['fdr_next'].notna().astype(int)
df['fdr_next'] = df['fdr_next'].fillna(3.0)  # neutral FDR for blanks
df['is_home_next'] = df['is_home_next'].fillna(0).astype(int)

print('fdr_next nulls after fill:', df['fdr_next'].isna().sum())
print('has_fixture distribution:', df['has_fixture'].value_counts().to_dict())

In [ ]:
# FDR next 3 gameweeks — average difficulty over the next 3 fixtures
fdr_rows = []
for offset in [1, 2, 3]:
    tmp = fixture_lookup.copy()
    tmp['next_round'] = tmp['round'] - offset + 1  # shift so round N looks ahead by offset
    tmp = tmp.rename(columns={'fdr': f'fdr_offset_{offset}', 'round': f'_round_{offset}'})
    fdr_rows.append(tmp[['next_round', 'team_id', f'fdr_offset_{offset}']])

# Simpler approach: compute per-player rolling future FDR
# For each player at round N, average fdr_next for rounds N, N+1, N+2
def fdr_next3(player_df, lookup):
    results = []
    for _, row in player_df.iterrows():
        future = lookup[
            (lookup['round'].isin([row['round'] + 1, row['round'] + 2, row['round'] + 3])) &
            (lookup['team_id'] == row['team'])
        ]['fdr']
        results.append(future.mean() if not future.empty else np.nan)
    return pd.Series(results, index=player_df.index)

df['fdr_next3'] = df.groupby('player_id', group_keys=False).apply(
    lambda g: fdr_next3(g, fixture_lookup)
).fillna(3.0)

print('fdr_next3 sample:')
print(df[['player_id', 'round', 'team', 'fdr_next', 'fdr_next3', 'is_home_next', 'has_fixture']].head(10).to_string(index=False))

In [ ]:
# Opponent goals conceded average — how leaky is the next opponent?
# Compute from actual match results in the fixture data
finished = fixtures[fixtures['finished'] == True].copy()

# Goals conceded from each team's perspective
home_conceded = finished[['event', 'team_h', 'team_a_score']].rename(
    columns={'event': 'round', 'team_h': 'team_id', 'team_a_score': 'goals_conceded_in_match'}
)
away_conceded = finished[['event', 'team_a', 'team_h_score']].rename(
    columns={'event': 'round', 'team_a': 'team_id', 'team_h_score': 'goals_conceded_in_match'}
)
team_conceded = pd.concat([home_conceded, away_conceded]).sort_values(['team_id', 'round'])
team_conceded['goals_conceded_in_match'] = pd.to_numeric(
    team_conceded['goals_conceded_in_match'], errors='coerce'
).fillna(0)

# Rolling 5-game average goals conceded per team (shifted to avoid leakage)
team_conceded['opp_goals_conceded_avg'] = team_conceded.groupby('team_id')['goals_conceded_in_match'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)

# The opponent's conceded avg relevant to our player is from the NEXT round
opp_lookup = team_conceded[['round', 'team_id', 'opp_goals_conceded_avg']].rename(
    columns={'round': 'next_round', 'team_id': 'next_opponent_id'}
)

df = df.merge(opp_lookup, on=['next_round', 'next_opponent_id'], how='left')
df['opp_goals_conceded_avg'] = df['opp_goals_conceded_avg'].fillna(
    df['opp_goals_conceded_avg'].median()
)

print('Opponent leakiness feature done.')
print(df[['player_id', 'round', 'next_opponent_id', 'opp_goals_conceded_avg']].head(10).to_string(index=False))

## 6. Value features

In [ ]:
# Price change over last 3 gameweeks
df['price_3gw_ago'] = df.groupby('player_id')['value'].shift(3)
df['price_change_3gw'] = df['value'] - df['price_3gw_ago']
df['price_change_3gw'] = df['price_change_3gw'].fillna(0)
df.drop(columns=['price_3gw_ago'], inplace=True)

# Points per million (value metric)
df['pts_per_million'] = df['rolling_pts_5gw'] / df['value'].replace(0, np.nan)
df['pts_per_million'] = df['pts_per_million'].fillna(0)

# Ownership percentage (selected is raw count, convert to %)
# Total FPL players is in bootstrap — use a reasonable constant (13M from the API)
TOTAL_FPL_PLAYERS = 13_038_826
df['ownership_pct'] = (df['selected'] / TOTAL_FPL_PLAYERS * 100).round(2)

# Ownership change over 3 GWs
df['ownership_3gw_ago'] = df.groupby('player_id')['ownership_pct'].shift(3)
df['ownership_change_3gw'] = df['ownership_pct'] - df['ownership_3gw_ago']
df['ownership_change_3gw'] = df['ownership_change_3gw'].fillna(0)
df.drop(columns=['ownership_3gw_ago'], inplace=True)

# Differential flag: owned by < 10% of managers
df['is_differential'] = (df['ownership_pct'] < 10).astype(int)

# Season context
df['gw_number'] = df['round']
df['games_played'] = df.groupby('player_id').cumcount()  # 0-indexed games played so far

print('Value features done.')
print(df[['player_id', 'round', 'value', 'price_change_3gw', 'pts_per_million',
          'ownership_pct', 'ownership_change_3gw', 'is_differential']].head(10).to_string(index=False))

## 7. Positional encoding

In [ ]:
for pos in ['GKP', 'DEF', 'MID', 'FWD']:
    df[f'is_{pos.lower()}'] = (df['position'] == pos).astype(int)

print('Position dummies:')
print(df[['position', 'is_gkp', 'is_def', 'is_mid', 'is_fwd']].drop_duplicates().sort_values('position'))

## 8. Target variable

`pts_next_gw` — what did this player score in the **next** gameweek.
Built by shifting `total_points` back by 1 within each player group.

In [ ]:
df['pts_next_gw'] = df.groupby('player_id')['total_points'].shift(-1)

print('Target distribution (players with a next GW):')
print(df['pts_next_gw'].describe())
print()
print('Null target rows (last GW per player — will be dropped):', df['pts_next_gw'].isna().sum())

## 9. Final dataset — filter and select columns

In [ ]:
FEATURE_COLS = [
    # Identity
    'player_id', 'web_name', 'round', 'position', 'team',
    # Form
    'rolling_pts_3gw', 'rolling_pts_5gw',
    'rolling_minutes_3gw', 'rolling_minutes_5gw',
    'minutes_consistency', 'blank_gw_flag', 'form_streak',
    # xG/xA
    'rolling_xg_3gw', 'rolling_xg_5gw',
    'rolling_xa_3gw', 'rolling_xa_5gw',
    'xg_overperformance', 'shots_per_90', 'key_passes_per_90',
    # Fixture
    'fdr_next', 'fdr_next3', 'is_home_next',
    'opp_goals_conceded_avg', 'has_fixture',
    # Value
    'value', 'price_change_3gw', 'pts_per_million',
    'ownership_pct', 'ownership_change_3gw', 'is_differential',
    # Context
    'gw_number', 'games_played',
    # Position dummies
    'is_gkp', 'is_def', 'is_mid', 'is_fwd',
    # Target
    'pts_next_gw',
]

features = df[FEATURE_COLS].copy()

# Drop rows with no target (last GW per player)
features = features.dropna(subset=['pts_next_gw'])

# Drop rows with no rolling history (first GW — rolling_pts_3gw will be NaN)
features = features.dropna(subset=['rolling_pts_3gw'])

print('Final feature set shape:', features.shape)
print('Null counts per column:')
null_counts = features.isna().sum()
print(null_counts[null_counts > 0])

## 10. Validation

In [ ]:
# 1. Leakage check: rolling_pts_3gw should correlate with pts_next_gw but not be identical
corr = features[['rolling_pts_3gw', 'rolling_pts_5gw', 'pts_next_gw']].corr()
print('Correlation matrix (expect 0.3-0.6 between rolling and target):')
print(corr.round(3))
print()

# 2. Per-player sanity check — verify shift is correct
pid = features['player_id'].iloc[0]
sample = df[df['player_id'] == pid][['round', 'total_points', 'rolling_pts_3gw', 'pts_next_gw']].head(8)
print(f'Player {pid} — rolling and target check:')
print(sample.to_string(index=False))

In [ ]:
# 3. Target distribution
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
features['pts_next_gw'].clip(0, 20).hist(bins=21, ax=ax, edgecolor='black')
ax.set_xlabel('pts_next_gw')
ax.set_ylabel('Count')
ax.set_title('Target variable distribution (clipped at 20)')
plt.tight_layout()
plt.show()

print('Mean target:', features['pts_next_gw'].mean().round(3))
print('Median target:', features['pts_next_gw'].median())
print('% of rows with 0 pts:', (features['pts_next_gw'] == 0).mean().round(3))

In [ ]:
# 4. Feature coverage summary
model_features = [c for c in FEATURE_COLS
                  if c not in ['player_id', 'web_name', 'round', 'position', 'team', 'pts_next_gw']]
print(f'Total model features: {len(model_features)}')
print(f'Training rows: {len(features)}')
print()
print('Rows by position:')
print(features['position'].value_counts())

## 11. Save

In [ ]:
features.to_parquet(PROCESSED / 'features.parquet', index=False)

print('Saved features.parquet')
print(f'  Rows:     {len(features)}')
print(f'  Columns:  {len(features.columns)}')
print(f'  Players:  {features["player_id"].nunique()}')
print(f'  GWs:      {features["round"].nunique()} (rounds {features["round"].min()}–{features["round"].max()})')